In [4]:
from __future__ import annotations

import json
import re
import warnings
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning)

# ============================================================
# 00. CONFIG
# ============================================================
BASE = Path('/mnt/data')
OUT = BASE / '00_cleanup_eda_outputs'
FIG = OUT / 'figures'
TAB = OUT / 'tables'
CLEAN = OUT / 'cleaned'
for p in (OUT, FIG, TAB, CLEAN):
    p.mkdir(parents=True, exist_ok=True)

FILES = {
    'labs': BASE / 'labs_deidentified.xlsx',
    'meds': BASE / 'medications_deidentified.xlsx',
    'notes': BASE / 'notes_deidentified.xlsx',
    'mm': BASE / 'multimodal_patient_year.xlsx',
}

# Optional synthetic files. The script searches these patterns automatically.
SYN_FULL_PATTERNS = [
    '**/*synthetic*FULL*.csv',
    '**/*synthetic*full*.csv',
]
SYN_MODEL_PATTERNS = [
    '**/*synthetic*MODEL_READY*.csv',
    '**/*synthetic*model_ready*.csv',
    '**/*MODEL_READY*.csv',
]

# Do NOT automatically remove clinical rows merely because they look odd.
DROP_EXACT_MED_DUPLICATES = False
DROP_EXACT_OTHER_DUPLICATES = False

# Minimum group size before robust outlier flagging for lab value/unit groups.
MIN_OUTLIER_GROUP_N = 8

# ============================================================
# 01. HELPERS
# ============================================================
def banner(txt: str):
    print('\n' + '=' * 86)
    print(txt)
    print('=' * 86)


def savefig(name: str):
    plt.tight_layout()
    plt.savefig(FIG / name, dpi=160, bbox_inches='tight')
    plt.close()


def clean_text_series(s: pd.Series) -> pd.Series:
    """Conservative categorical normalization: trim + collapse whitespace."""
    out = s.astype('string')
    out = out.str.strip().str.replace(r'\s+', ' ', regex=True)
    out = out.replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA, '<NA>': pd.NA})
    return out


def normalize_patient_id(s: pd.Series) -> pd.Series:
    """Normalize only formatting, never renumber patients."""
    out = clean_text_series(s)
    # Preserve Patient_001 style while standardizing common whitespace/hyphen variants.
    out = out.str.replace(r'^patient[\s\-]*', 'Patient_', regex=True, case=False)
    out = out.str.replace(r'^Patient__+', 'Patient_', regex=True)
    return out


def coerce_year(s: pd.Series) -> pd.Series:
    """Extract a plausible 4-digit year; keeps NA if no valid year."""
    if pd.api.types.is_numeric_dtype(s):
        y = pd.to_numeric(s, errors='coerce')
    else:
        y = pd.to_numeric(s.astype('string').str.extract(r'((?:19|20)\d{2})', expand=False), errors='coerce')
    y = y.where(y.between(1900, 2100))
    return y.astype('Int64')


def missingness_table(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    out = pd.DataFrame({
        'column': df.columns,
        'missing_n': [int(df[c].isna().sum()) for c in df.columns],
        'missing_pct': [float(df[c].isna().mean() * 100) for c in df.columns],
        'unique_non_null': [int(df[c].nunique(dropna=True)) for c in df.columns],
        'dtype': [str(df[c].dtype) for c in df.columns],
    })
    return out.sort_values(['missing_pct', 'column'], ascending=[False, True]).reset_index(drop=True)


def plot_missingness(df: pd.DataFrame, title: str, filename: str, top_n: int = 40):
    miss = (df.isna().mean() * 100).sort_values(ascending=False).head(top_n)
    fig_h = max(4, 0.28 * len(miss) + 1.5)
    plt.figure(figsize=(9, fig_h))
    plt.barh(miss.index[::-1], miss.values[::-1])
    plt.xlabel('Missing values (%)')
    plt.title(title)
    plt.xlim(0, 100)
    savefig(filename)


def exact_duplicate_report(df: pd.DataFrame, name: str) -> dict:
    dup_all = df.duplicated(keep=False)
    dup_extra = df.duplicated(keep='first')
    report = {
        'rows': int(len(df)),
        'exact_duplicate_rows_in_groups': int(dup_all.sum()),
        'exact_duplicate_extra_rows': int(dup_extra.sum()),
        'exact_duplicate_pct_extra': float(dup_extra.mean() * 100 if len(df) else 0),
    }
    if dup_all.any():
        df.loc[dup_all].sort_values(list(df.columns), kind='stable').to_csv(
            TAB / f'{name}_exact_duplicate_rows.csv', index=False
        )
    return report


def robust_outlier_mask(x: pd.Series, min_n: int = MIN_OUTLIER_GROUP_N) -> pd.Series:
    """Robust extreme flag: median +/- 5*MAD, fallback 3*IQR if MAD==0."""
    x = pd.to_numeric(x, errors='coerce')
    mask = pd.Series(False, index=x.index)
    valid = x.dropna()
    if len(valid) < min_n:
        return mask
    med = valid.median()
    mad = np.median(np.abs(valid - med))
    if mad > 0:
        # 1.4826 converts MAD to a std-like scale under normality.
        z = np.abs(x - med) / (1.4826 * mad)
        return (z > 5).fillna(False)
    q1, q3 = valid.quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr > 0:
        return ((x < q1 - 3 * iqr) | (x > q3 + 3 * iqr)).fillna(False)
    return mask


def find_first_existing(patterns: Iterable[str]) -> Path | None:
    hits = []
    for pat in patterns:
        hits.extend(BASE.glob(pat))
    hits = [p for p in hits if p.is_file() and '00_cleanup_eda_outputs' not in str(p)]
    return sorted(set(hits))[0] if hits else None


def infer_patient_col(df: pd.DataFrame) -> str | None:
    candidates = ['Patient', 'patient', 'patient_id', 'Patient_ID', 'Profile Key', 'profile_key']
    for c in candidates:
        if c in df.columns:
            return c
    for c in df.columns:
        if 'patient' in c.lower() or 'profile key' in c.lower():
            return c
    return None


def infer_year_col(df: pd.DataFrame) -> str | None:
    preferred = ['Year', 'year', 'Result Date', 'Service Date', 'Start Date', 'Index_Date']
    for c in preferred:
        if c in df.columns:
            return c
    for c in df.columns:
        if 'year' in c.lower():
            return c
    return None


def save_value_counts(s: pd.Series, path: Path, top_n: int | None = None):
    vc = s.value_counts(dropna=False).rename_axis('value').reset_index(name='count')
    if top_n is not None:
        vc = vc.head(top_n)
    vc.to_csv(path, index=False)


# ============================================================
# 02. LOAD DATA
# ============================================================
banner('LOAD SOURCE DATA')
for k, p in FILES.items():
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

labs = pd.read_excel(FILES['labs'])
meds = pd.read_excel(FILES['meds'])
notes = pd.read_excel(FILES['notes'])
mm = pd.read_csv(FILES['mm'])

print('labs :', labs.shape)
print('meds :', meds.shape)
print('notes:', notes.shape)
print('mm   :', mm.shape)

summary: dict = {'source_files': {}, 'synthetic': {}}

# ============================================================
# 03. CLEANUP / QUALITY AUDIT: LABS
# ============================================================
banner('CLEANUP / QUALITY AUDIT — LABS')
labs_c = labs.copy()

if 'Patient' in labs_c:
    labs_c['Patient'] = normalize_patient_id(labs_c['Patient'])

# Preserve original value and create an explicitly numeric parsed copy.
if 'Numeric Value' in labs_c:
    labs_c['Numeric Value_raw'] = labs_c['Numeric Value']
    labs_c['Numeric Value'] = pd.to_numeric(labs_c['Numeric Value'], errors='coerce')

if 'Result Date' in labs_c:
    labs_c['Result_Year'] = coerce_year(labs_c['Result Date'])

# Normalize selected categorical text columns conservatively.
for c in ['Lab Component Name', 'Lab Component Base Name', 'Lab Component Common Name',
          'Lab Abbreviation', 'Lab Type', 'Loinc Code', 'Loinc Name', 'Flag', 'Unit']:
    if c in labs_c:
        labs_c[c] = clean_text_series(labs_c[c])

lab_dup = exact_duplicate_report(labs_c.drop(columns=['Numeric Value_raw'], errors='ignore'), 'labs')

# Identify analyte-unit pairs and flag robust extremes, but DO NOT delete them.
lab_name_col = next((c for c in ['Lab Component Common Name', 'Lab Component Name', 'Lab Component Base Name'] if c in labs_c), None)
labs_c['robust_extreme_flag'] = False
if lab_name_col and 'Numeric Value' in labs_c:
    group_cols = [lab_name_col] + (['Unit'] if 'Unit' in labs_c else [])
    for _, idx in labs_c.groupby(group_cols, dropna=False).groups.items():
        labs_c.loc[idx, 'robust_extreme_flag'] = robust_outlier_mask(labs_c.loc[idx, 'Numeric Value']).values

# Unit consistency per analyte.
if lab_name_col and 'Unit' in labs_c:
    unit_report = (
        labs_c.dropna(subset=[lab_name_col])
              .groupby(lab_name_col, dropna=False)['Unit']
              .agg(n_units=lambda x: x.dropna().nunique(), units=lambda x: ' | '.join(sorted(map(str, x.dropna().unique()))[:25]))
              .sort_values('n_units', ascending=False)
              .reset_index()
    )
    unit_report.to_csv(TAB / 'labs_unit_consistency.csv', index=False)

# Numeric lab summary by analyte + unit.
if lab_name_col and 'Numeric Value' in labs_c:
    lab_group = [lab_name_col] + (['Unit'] if 'Unit' in labs_c else [])
    lab_stats = (
        labs_c.dropna(subset=['Numeric Value'])
              .groupby(lab_group, dropna=False)['Numeric Value']
              .agg(n='count', mean='mean', std='std', min='min', q1=lambda x: x.quantile(.25),
                   median='median', q3=lambda x: x.quantile(.75), max='max')
              .reset_index()
              .sort_values('n', ascending=False)
    )
    lab_stats.to_csv(TAB / 'labs_numeric_summary_by_analyte_unit.csv', index=False)

labs_c.to_csv(CLEAN / 'labs_cleaned_flagged.csv', index=False)
missingness_table(labs_c).to_csv(TAB / 'labs_missingness.csv', index=False)
plot_missingness(labs_c, 'Labs — missingness', 'labs_missingness.png')

summary['source_files']['labs'] = {
    **lab_dup,
    'unique_patients': int(labs_c['Patient'].nunique()) if 'Patient' in labs_c else None,
    'extreme_rows_flagged': int(labs_c['robust_extreme_flag'].sum()),
}
print(summary['source_files']['labs'])

# ============================================================
# 04. CLEANUP / QUALITY AUDIT: MEDICATIONS
# ============================================================
banner('CLEANUP / QUALITY AUDIT — MEDICATIONS')
meds_c = meds.copy()
if 'Patient' in meds_c:
    meds_c['Patient'] = normalize_patient_id(meds_c['Patient'])

for c in meds_c.columns:
    if meds_c[c].dtype == 'object':
        meds_c[c] = clean_text_series(meds_c[c])

for c in ['Start Date', 'Administration Date', 'End Date', 'Discontinued Date']:
    if c in meds_c:
        meds_c[c + '_Year'] = coerce_year(meds_c[c])

med_dup = exact_duplicate_report(meds_c, 'medications')

# Investigate duplicates rather than assuming they are errors.
meds_c['exact_duplicate_group'] = meds_c.duplicated(keep=False)
meds_c['exact_duplicate_extra'] = meds_c.duplicated(keep='first')

# Report duplicate multiplicity.
if meds_c['exact_duplicate_group'].any():
    dup_counts = (
        meds_c.loc[meds_c['exact_duplicate_group']]
              .groupby(list(meds_c.columns.drop(['exact_duplicate_group', 'exact_duplicate_extra'])), dropna=False)
              .size().reset_index(name='multiplicity')
              .sort_values('multiplicity', ascending=False)
    )
    dup_counts.to_csv(TAB / 'medications_duplicate_groups_with_multiplicity.csv', index=False)

# Optional exact deduplication; default False.
if DROP_EXACT_MED_DUPLICATES:
    meds_clean_export = meds_c.loc[~meds_c['exact_duplicate_extra']].copy()
else:
    meds_clean_export = meds_c.copy()

meds_clean_export.to_csv(CLEAN / 'medications_cleaned_flagged.csv', index=False)
missingness_table(meds_c).to_csv(TAB / 'medications_missingness.csv', index=False)
plot_missingness(meds_c, 'Medications — missingness', 'medications_missingness.png')

summary['source_files']['medications'] = {
    **med_dup,
    'unique_patients': int(meds_c['Patient'].nunique()) if 'Patient' in meds_c else None,
    'drop_exact_duplicates_enabled': DROP_EXACT_MED_DUPLICATES,
}
print(summary['source_files']['medications'])

# ============================================================
# 05. CLEANUP / QUALITY AUDIT: NOTES
# ============================================================
banner('CLEANUP / QUALITY AUDIT — NOTES')
notes_c = notes.copy()
profile_col = 'Profile Key' if 'Profile Key' in notes_c else infer_patient_col(notes_c)
if profile_col:
    notes_c[profile_col] = normalize_patient_id(notes_c[profile_col])

for c in notes_c.columns:
    if notes_c[c].dtype == 'object' and c != 'Notes':
        notes_c[c] = clean_text_series(notes_c[c])

for c in ['Service Date', 'Creation Date', 'Last Edited Date']:
    if c in notes_c:
        notes_c[c + '_Year'] = coerce_year(notes_c[c])

notes_dup = exact_duplicate_report(notes_c, 'notes')
notes_c.to_csv(CLEAN / 'notes_cleaned.csv', index=False)
missingness_table(notes_c).to_csv(TAB / 'notes_missingness.csv', index=False)
plot_missingness(notes_c, 'Notes — missingness', 'notes_missingness.png')

summary['source_files']['notes'] = {
    **notes_dup,
    'unique_patients': int(notes_c[profile_col].nunique()) if profile_col else None,
}
print(summary['source_files']['notes'])

# ============================================================
# 06. CLEANUP / QUALITY AUDIT: MULTIMODAL PATIENT-YEAR
# ============================================================
banner('CLEANUP / QUALITY AUDIT — MULTIMODAL PATIENT-YEAR')
mm_c = mm.copy()
mm_patient = infer_patient_col(mm_c)
if mm_patient:
    mm_c[mm_patient] = normalize_patient_id(mm_c[mm_patient])

# Normalize object/categorical columns except long free-text fields.
for c in mm_c.columns:
    if mm_c[c].dtype == 'object' and 'note' not in c.lower() and 'text' not in c.lower():
        mm_c[c] = clean_text_series(mm_c[c])

# Make likely year/date fields usable.
for c in list(mm_c.columns):
    if c.lower() == 'year' or c.lower().endswith('_year'):
        mm_c[c] = coerce_year(mm_c[c])

mm_dup = exact_duplicate_report(mm_c, 'multimodal_patient_year')
mm_c.to_csv(CLEAN / 'multimodal_patient_year_cleaned.csv', index=False)
missingness_table(mm_c).to_csv(TAB / 'multimodal_missingness.csv', index=False)
plot_missingness(mm_c, 'Multimodal patient-year — missingness (top 40)', 'multimodal_missingness_top40.png')

summary['source_files']['multimodal_patient_year'] = {
    **mm_dup,
    'unique_patients': int(mm_c[mm_patient].nunique()) if mm_patient else None,
}
print(summary['source_files']['multimodal_patient_year'])

# ============================================================
# 07. REAL/SOURCE EDA
# ============================================================
banner('EDA — SOURCE / REAL DATA')

# ---- 7A. Patient overlap by modality ----
patient_sets = {}
if 'Patient' in labs_c:
    patient_sets['labs'] = set(labs_c['Patient'].dropna())
if 'Patient' in meds_c:
    patient_sets['medications'] = set(meds_c['Patient'].dropna())
if profile_col:
    patient_sets['notes'] = set(notes_c[profile_col].dropna())
if mm_patient:
    patient_sets['multimodal'] = set(mm_c[mm_patient].dropna())

all_patients = sorted(set().union(*patient_sets.values())) if patient_sets else []
overlap_rows = []
for p in all_patients:
    row = {'Patient': p}
    for name, st in patient_sets.items():
        row[name] = int(p in st)
    overlap_rows.append(row)
overlap = pd.DataFrame(overlap_rows)
overlap.to_csv(TAB / 'patient_modality_overlap.csv', index=False)

modality_counts = pd.DataFrame({
    'modality': list(patient_sets.keys()),
    'n_patients': [len(s) for s in patient_sets.values()]
})
modality_counts.to_csv(TAB / 'patients_per_modality.csv', index=False)
plt.figure(figsize=(7, 4))
plt.bar(modality_counts['modality'], modality_counts['n_patients'])
plt.ylabel('Unique patients')
plt.title('Patients represented in each source')
savefig('patients_per_modality.png')

# Pairwise overlap matrix.
mods = list(patient_sets)
mat = pd.DataFrame(index=mods, columns=mods, dtype=int)
for a in mods:
    for b in mods:
        mat.loc[a, b] = len(patient_sets[a] & patient_sets[b])
mat.to_csv(TAB / 'patient_pairwise_overlap_matrix.csv')

# ---- 7B. Observations per patient ----
def observations_per_patient(df: pd.DataFrame, patient_col: str, name: str):
    counts = df.groupby(patient_col).size().rename('n_observations').sort_values(ascending=False)
    counts.to_csv(TAB / f'{name}_observations_per_patient.csv')
    plt.figure(figsize=(7, 4))
    plt.hist(counts.values, bins=min(30, max(5, int(np.sqrt(len(counts))))))
    plt.xlabel('Observations per patient')
    plt.ylabel('Patients')
    plt.title(f'{name}: observations per patient')
    savefig(f'{name}_observations_per_patient.png')
    return counts

lab_obs = observations_per_patient(labs_c, 'Patient', 'labs') if 'Patient' in labs_c else None
med_obs = observations_per_patient(meds_c, 'Patient', 'medications') if 'Patient' in meds_c else None
notes_obs = observations_per_patient(notes_c, profile_col, 'notes') if profile_col else None
mm_obs = observations_per_patient(mm_c, mm_patient, 'multimodal') if mm_patient else None

# ---- 7C. Observations per year ----
def plot_obs_by_year(df: pd.DataFrame, year_col: str, name: str):
    if year_col not in df:
        return
    d = df.dropna(subset=[year_col]).groupby(year_col).size().rename('n').reset_index()
    if d.empty:
        return
    d.to_csv(TAB / f'{name}_observations_per_year.csv', index=False)
    plt.figure(figsize=(9, 4))
    plt.plot(d[year_col].astype(int), d['n'], marker='o')
    plt.xlabel('Year')
    plt.ylabel('Observations')
    plt.title(f'{name}: observations per year')
    savefig(f'{name}_observations_per_year.png')

if 'Result_Year' in labs_c:
    plot_obs_by_year(labs_c, 'Result_Year', 'labs')
if 'Start Date_Year' in meds_c:
    plot_obs_by_year(meds_c, 'Start Date_Year', 'medications_start')
if 'Service Date_Year' in notes_c:
    plot_obs_by_year(notes_c, 'Service Date_Year', 'notes_service')
if 'Year' in mm_c:
    plot_obs_by_year(mm_c, 'Year', 'multimodal')

# ---- 7D. Lab distributions ----
if lab_name_col and 'Numeric Value' in labs_c:
    top_labs = (
        labs_c.dropna(subset=['Numeric Value'])
              .groupby(lab_name_col).size().sort_values(ascending=False).head(12).index
    )
    for lab_name in top_labs:
        sub = labs_c.loc[(labs_c[lab_name_col] == lab_name) & labs_c['Numeric Value'].notna(), ['Numeric Value', 'Unit']].copy()
        # Use the most common unit so incompatible units are not mixed in one histogram.
        if 'Unit' in sub and sub['Unit'].notna().any():
            unit = sub['Unit'].mode(dropna=True).iloc[0]
            vals = sub.loc[sub['Unit'] == unit, 'Numeric Value']
        else:
            unit = None
            vals = sub['Numeric Value']
        if len(vals) < 5:
            continue
        plt.figure(figsize=(7, 4))
        plt.hist(vals, bins=30)
        plt.xlabel(f'Numeric value' + (f' ({unit})' if unit else ''))
        plt.ylabel('Count')
        plt.title(str(lab_name))
        safe = re.sub(r'[^A-Za-z0-9_-]+', '_', str(lab_name))[:80]
        savefig(f'lab_distribution_{safe}.png')

# ---- 7E. Medication class / generic frequencies ----
for c, fn, ttl in [
    ('Medication Therapeutic Class', 'medication_therapeutic_class_top30.csv', 'Top medication therapeutic classes'),
    ('Medication Pharmaceutical Class', 'medication_pharmaceutical_class_top30.csv', 'Top medication pharmaceutical classes'),
    ('Simple Generic Name', 'medication_generic_top30.csv', 'Top generic medications'),
]:
    if c in meds_c:
        vc = meds_c[c].value_counts().head(30)
        vc.rename('count').to_csv(TAB / fn)
        plt.figure(figsize=(9, max(5, 0.25 * len(vc))))
        plt.barh(vc.index[::-1], vc.values[::-1])
        plt.xlabel('Rows')
        plt.title(ttl)
        savefig(fn.replace('.csv', '.png'))

# Patient-level medication prevalence (more useful than raw-row frequency).
med_id_col = 'Simple Generic Name' if 'Simple Generic Name' in meds_c else None
if med_id_col and 'Patient' in meds_c:
    prev = (meds_c.dropna(subset=[med_id_col]).groupby(med_id_col)['Patient'].nunique() / meds_c['Patient'].nunique())
    prev = prev.sort_values(ascending=False).rename('patient_prevalence')
    prev.to_csv(TAB / 'medication_patient_prevalence.csv')

# ---- 7F. Follow-up and longitudinal sparsity in multimodal table ----
if mm_patient and 'Year' in mm_c:
    follow = (
        mm_c.dropna(subset=['Year'])
            .groupby(mm_patient)['Year']
            .agg(first_year='min', last_year='max', n_observed_year_rows='size', n_unique_years='nunique')
    )
    follow['followup_span_years'] = follow['last_year'] - follow['first_year']
    follow.to_csv(TAB / 'multimodal_followup_by_patient.csv')

    plt.figure(figsize=(7, 4))
    plt.hist(follow['n_unique_years'], bins=range(1, int(follow['n_unique_years'].max()) + 2))
    plt.xlabel('Unique observed years per patient')
    plt.ylabel('Patients')
    plt.title('Longitudinal coverage')
    savefig('multimodal_unique_years_per_patient.png')

# Modality-style feature groups in multimodal table.
feature_groups = {
    'lab': [c for c in mm_c.columns if c.lower().startswith('lab__')],
    'med': [c for c in mm_c.columns if c.lower().startswith('med__')],
    'note': [c for c in mm_c.columns if c.lower().startswith('note_') or c.lower().startswith('note__')],
}
sparsity_rows = []
for group, cols in feature_groups.items():
    if cols:
        vals = mm_c[cols]
        sparsity_rows.append({
            'feature_group': group,
            'n_columns': len(cols),
            'missing_pct_cells': float(vals.isna().mean().mean() * 100),
            'rows_with_any_nonmissing_pct': float(vals.notna().any(axis=1).mean() * 100),
        })
pd.DataFrame(sparsity_rows).to_csv(TAB / 'multimodal_feature_group_sparsity.csv', index=False)

# ---- 7G. Valve failure distribution ----
valve_failure_col = next((c for c in mm_c.columns if c.lower() == 'valve_failure'), None)
if valve_failure_col:
    vf = mm_c[valve_failure_col].value_counts(dropna=False).rename_axis('Valve_Failure').reset_index(name='rows')
    vf.to_csv(TAB / 'valve_failure_distribution_rows.csv', index=False)
    plt.figure(figsize=(6, 4))
    labels = vf['Valve_Failure'].astype('string').fillna('NA')
    plt.bar(labels, vf['rows'])
    plt.xlabel('Valve_Failure')
    plt.ylabel('Rows')
    plt.title('Valve_Failure distribution')
    savefig('valve_failure_distribution_rows.png')

    if mm_patient:
        # Patient-level max event flag avoids counting repeated event rows as separate patients.
        patient_vf = pd.to_numeric(mm_c[valve_failure_col], errors='coerce').groupby(mm_c[mm_patient]).max()
        patient_vf.value_counts(dropna=False).rename('patients').to_csv(TAB / 'valve_failure_distribution_patients.csv')

# ---- 7H. Index-date / time relation ----
index_cols = [c for c in mm_c.columns if c.lower() in {'index_date', 'index_year'} or 'index_date' in c.lower()]
if index_cols:
    # Do not force a date interpretation; save descriptives for audit.
    for c in index_cols:
        pd.DataFrame({'value': mm_c[c]}).describe(include='all').to_csv(TAB / f'{c}_description.csv')

# ---- 7I. Potential leakage audit ----
leak_patterns = [
    r'failure', r'event', r'outcome', r'death', r'mortality', r'phenotype',
    r'prosthetic.*failure', r'is_event_row', r'duration', r'time_to', r'target'
]
leak_re = re.compile('|'.join(leak_patterns), flags=re.I)
leak_cols = [c for c in mm_c.columns if leak_re.search(c)]
pd.DataFrame({'potential_leakage_column': leak_cols}).to_csv(TAB / 'potential_leakage_columns.csv', index=False)

summary['real_eda'] = {
    'patients_per_modality': {k: len(v) for k, v in patient_sets.items()},
    'pairwise_overlap': mat.to_dict() if patient_sets else {},
    'potential_leakage_columns': leak_cols,
    'feature_group_sparsity': sparsity_rows,
}

# ============================================================
# 08. OPTIONAL SYNTHETIC DATA EDA / VALIDATION
# ============================================================
banner('EDA / VALIDATION — SYNTHETIC DATA (IF PRESENT)')
syn_full_path = find_first_existing(SYN_FULL_PATTERNS)
syn_model_path = find_first_existing(SYN_MODEL_PATTERNS)

print('Synthetic FULL      :', syn_full_path)
print('Synthetic MODEL_READY:', syn_model_path)

syn_full = pd.read_csv(syn_full_path) if syn_full_path else None
syn_model = pd.read_csv(syn_model_path) if syn_model_path else None

# Prefer FULL for generator diagnostics, model-ready for leakage-safe feature comparisons.
syn = syn_full if syn_full is not None else syn_model

if syn is None:
    print('\nNo synthetic CSV found. Source cleanup/EDA is complete.')
    print('When the generator output CSV is placed under /mnt/data, rerun this same script.')
    summary['synthetic']['status'] = 'not_found'
else:
    syn_c = syn.copy()
    syn_patient = infer_patient_col(syn_c)
    if syn_patient:
        syn_c[syn_patient] = normalize_patient_id(syn_c[syn_patient])

    # Generic missingness and duplicate checks.
    missingness_table(syn_c).to_csv(TAB / 'synthetic_missingness.csv', index=False)
    plot_missingness(syn_c, 'Synthetic cohort — missingness (top 40)', 'synthetic_missingness_top40.png')
    syn_dup = exact_duplicate_report(syn_c, 'synthetic')

    # Locate likely survival columns.
    lower_map = {c.lower(): c for c in syn_c.columns}
    event_col = next((lower_map[k] for k in lower_map if k in {'event', 'event_observed', 'valve_failure'}), None)
    duration_col = next((c for c in syn_c.columns if c.lower() in {'duration', 'duration_months', 'followup_months', 'time_to_event_months'}), None)
    phenotype_col = next((c for c in syn_c.columns if 'phenotype' in c.lower()), None)
    year_col = infer_year_col(syn_c)

    syn_metrics = {**syn_dup}
    if syn_patient:
        syn_metrics['unique_patients'] = int(syn_c[syn_patient].nunique())

    # Event/censoring at patient level where possible.
    if event_col:
        ev = pd.to_numeric(syn_c[event_col], errors='coerce')
        if syn_patient:
            evp = ev.groupby(syn_c[syn_patient]).max()
        else:
            evp = ev
        event_rate = float(evp.mean())
        syn_metrics['event_rate'] = event_rate
        syn_metrics['censoring_rate'] = float(1 - event_rate)
        pd.DataFrame({'event': evp}).to_csv(TAB / 'synthetic_patient_event_status.csv')

    # Duration distribution.
    if duration_col:
        dur = pd.to_numeric(syn_c[duration_col], errors='coerce')
        if syn_patient:
            durp = dur.groupby(syn_c[syn_patient]).max()
        else:
            durp = dur
        durp.describe().to_csv(TAB / 'synthetic_duration_summary.csv')
        plt.figure(figsize=(7, 4))
        plt.hist(durp.dropna(), bins=30)
        plt.xlabel(duration_col)
        plt.ylabel('Patients')
        plt.title('Synthetic duration distribution')
        savefig('synthetic_duration_distribution.png')

    # Phenotype distribution.
    if phenotype_col:
        if syn_patient:
            ph = syn_c[[syn_patient, phenotype_col]].drop_duplicates(subset=[syn_patient])[phenotype_col]
        else:
            ph = syn_c[phenotype_col]
        ph.value_counts(dropna=False).rename('patients').to_csv(TAB / 'synthetic_phenotype_distribution.csv')

    # Demographics / valve categories / comorbidities.
    interesting_exact = [
        'age_at_implant', 'age', 'sex', 'gender', 'valve_size_mm', 'valve_model',
        'valve_position', 'procedure_type', 'ckd', 'af', 'heart_failure', 'diabetes'
    ]
    for c in syn_c.columns:
        if c.lower() in interesting_exact:
            if pd.api.types.is_numeric_dtype(syn_c[c]) and syn_c[c].nunique(dropna=True) > 10:
                syn_c[c].describe().to_csv(TAB / f'synthetic_{c}_summary.csv')
                plt.figure(figsize=(7, 4))
                plt.hist(pd.to_numeric(syn_c[c], errors='coerce').dropna(), bins=25)
                plt.xlabel(c)
                plt.ylabel('Rows')
                plt.title(f'Synthetic {c}')
                safe = re.sub(r'[^A-Za-z0-9_-]+', '_', c)
                savefig(f'synthetic_{safe}_distribution.png')
            else:
                save_value_counts(syn_c[c], TAB / f'synthetic_{c}_counts.csv', top_n=50)

    # Number of visits / rows per patient.
    if syn_patient:
        syn_visits = syn_c.groupby(syn_patient).size().rename('n_rows')
        syn_visits.to_csv(TAB / 'synthetic_rows_per_patient.csv')
        plt.figure(figsize=(7, 4))
        plt.hist(syn_visits, bins=min(30, max(5, int(np.sqrt(len(syn_visits))))))
        plt.xlabel('Rows / visits per patient')
        plt.ylabel('Patients')
        plt.title('Synthetic longitudinal density')
        savefig('synthetic_rows_per_patient.png')

    # Phenotype -> failure timing test, if both are available.
    if phenotype_col and duration_col and syn_patient:
        tmp = syn_c[[syn_patient, phenotype_col, duration_col] + ([event_col] if event_col else [])].copy()
        tmp[duration_col] = pd.to_numeric(tmp[duration_col], errors='coerce')
        agg = {phenotype_col: 'first', duration_col: 'max'}
        if event_col:
            agg[event_col] = 'max'
        pt = tmp.groupby(syn_patient).agg(agg).reset_index()
        pt.groupby(phenotype_col)[duration_col].describe().to_csv(TAB / 'synthetic_duration_by_phenotype.csv')

        plt.figure(figsize=(8, 4))
        groups = [g[duration_col].dropna().values for _, g in pt.groupby(phenotype_col)]
        labels = [str(k) for k, _ in pt.groupby(phenotype_col)]
        if groups:
            plt.boxplot(groups, labels=labels, showfliers=False)
            plt.ylabel(duration_col)
            plt.xticks(rotation=25, ha='right')
            plt.title('Failure/follow-up timing by synthetic phenotype')
            savefig('synthetic_duration_by_phenotype.png')

    # Hemodynamic trajectories (if columns exist).
    hemo_candidates = [
        c for c in syn_c.columns
        if any(k in c.lower() for k in ['mean_gradient', 'peak_velocity', 'effective_orifice', 'regurgitation_grade'])
    ]
    time_candidate = next((c for c in syn_c.columns if c.lower() in {'months_since_implant', 'time_months', 'followup_months', 'month'}), None)
    if phenotype_col and time_candidate:
        syn_c[time_candidate] = pd.to_numeric(syn_c[time_candidate], errors='coerce')
        for c in hemo_candidates[:8]:
            syn_c[c] = pd.to_numeric(syn_c[c], errors='coerce')
            d = syn_c.dropna(subset=[time_candidate, c, phenotype_col])
            if d.empty:
                continue
            # Bin time into 12-month intervals for readable mean trajectories.
            d = d.assign(time_bin=(np.floor(d[time_candidate] / 12) * 12).astype(int))
            traj = d.groupby([phenotype_col, 'time_bin'])[c].mean().reset_index()
            traj.to_csv(TAB / f'synthetic_trajectory_{c}.csv', index=False)
            plt.figure(figsize=(8, 5))
            for ph, g in traj.groupby(phenotype_col):
                plt.plot(g['time_bin'], g[c], marker='o', label=str(ph))
            plt.xlabel('Months since implant (12-month bins)')
            plt.ylabel(c)
            plt.title(f'Synthetic trajectory: {c}')
            plt.legend()
            safe = re.sub(r'[^A-Za-z0-9_-]+', '_', c)
            savefig(f'synthetic_trajectory_{safe}.png')

    # Increasing gradient as failure approaches, if event rows and time-to-event can be inferred.
    grad_col = next((c for c in syn_c.columns if 'mean_gradient' in c.lower()), None)
    if grad_col and duration_col and time_candidate and event_col:
        d = syn_c.copy()
        d[grad_col] = pd.to_numeric(d[grad_col], errors='coerce')
        d[duration_col] = pd.to_numeric(d[duration_col], errors='coerce')
        d[time_candidate] = pd.to_numeric(d[time_candidate], errors='coerce')
        d[event_col] = pd.to_numeric(d[event_col], errors='coerce')
        d = d[d[event_col] == 1].copy()
        d['months_to_event'] = d[duration_col] - d[time_candidate]
        d = d[(d['months_to_event'] >= 0) & d[grad_col].notna()]
        if not d.empty:
            d['months_to_event_bin'] = (np.floor(d['months_to_event'] / 12) * 12).astype(int)
            g = d.groupby('months_to_event_bin')[grad_col].mean().reset_index().sort_values('months_to_event_bin')
            g.to_csv(TAB / 'synthetic_mean_gradient_by_months_to_event.csv', index=False)
            plt.figure(figsize=(8, 4))
            plt.plot(g['months_to_event_bin'], g[grad_col], marker='o')
            plt.gca().invert_xaxis()
            plt.xlabel('Months before event')
            plt.ylabel(grad_col)
            plt.title('Mean gradient approaching failure')
            savefig('synthetic_mean_gradient_approaching_failure.png')

    summary['synthetic'].update(syn_metrics)
    summary['synthetic']['status'] = 'analyzed'
    summary['synthetic']['path_used'] = str(syn_full_path or syn_model_path)

# ============================================================
# 09. REAL vs SYNTHETIC COMPARISON
# ============================================================
banner('REAL vs SYNTHETIC COMPARISON (IF SYNTHETIC PRESENT)')
if syn is not None:
    # Use model-ready synthetic when available to mirror modelling feature space.
    syn_cmp = syn_model.copy() if syn_model is not None else syn.copy()
    syn_p = infer_patient_col(syn_cmp)

    # 9A. Rows / observations per patient.
    compare_density_rows = []
    if mm_patient:
        rc = mm_c.groupby(mm_patient).size()
        compare_density_rows.append({'dataset': 'real', 'n_patients': rc.size, 'mean_rows_per_patient': rc.mean(), 'median_rows_per_patient': rc.median()})
    if syn_p:
        sc = syn_cmp.groupby(syn_p).size()
        compare_density_rows.append({'dataset': 'synthetic', 'n_patients': sc.size, 'mean_rows_per_patient': sc.mean(), 'median_rows_per_patient': sc.median()})
    pd.DataFrame(compare_density_rows).to_csv(TAB / 'real_vs_synthetic_observations_per_patient.csv', index=False)

    if mm_patient and syn_p:
        plt.figure(figsize=(8, 4))
        plt.hist(mm_c.groupby(mm_patient).size(), bins=25, alpha=0.55, label='real')
        plt.hist(syn_cmp.groupby(syn_p).size(), bins=25, alpha=0.55, label='synthetic')
        plt.xlabel('Rows per patient')
        plt.ylabel('Patients')
        plt.title('Real vs synthetic longitudinal density')
        plt.legend()
        savefig('real_vs_synthetic_rows_per_patient.png')

    # 9B. Overall missingness comparison for common columns.
    common = sorted(set(mm_c.columns) & set(syn_cmp.columns))
    if common:
        miss_cmp = pd.DataFrame({
            'column': common,
            'real_missing_pct': [mm_c[c].isna().mean() * 100 for c in common],
            'synthetic_missing_pct': [syn_cmp[c].isna().mean() * 100 for c in common],
        })
        miss_cmp['abs_diff_pct_points'] = (miss_cmp['real_missing_pct'] - miss_cmp['synthetic_missing_pct']).abs()
        miss_cmp.sort_values('abs_diff_pct_points', ascending=False).to_csv(TAB / 'real_vs_synthetic_missingness_common_columns.csv', index=False)

    # 9C. Numeric distributions for common features; report quantiles rather than force every plot.
    common_numeric = [
        c for c in common
        if pd.api.types.is_numeric_dtype(mm_c[c]) and pd.api.types.is_numeric_dtype(syn_cmp[c])
        and mm_c[c].notna().sum() >= 10 and syn_cmp[c].notna().sum() >= 10
    ]
    rows = []
    for c in common_numeric:
        for label, df_ in [('real', mm_c), ('synthetic', syn_cmp)]:
            x = pd.to_numeric(df_[c], errors='coerce').dropna()
            rows.append({
                'feature': c, 'dataset': label, 'n': len(x), 'mean': x.mean(), 'std': x.std(),
                'q05': x.quantile(.05), 'q25': x.quantile(.25), 'median': x.median(),
                'q75': x.quantile(.75), 'q95': x.quantile(.95),
            })
    pd.DataFrame(rows).to_csv(TAB / 'real_vs_synthetic_common_numeric_summary.csv', index=False)

    # Plot a focused subset: lab__/hemodynamic features when common.
    focus = [c for c in common_numeric if c.lower().startswith('lab__') or any(k in c.lower() for k in ['gradient', 'velocity', 'orifice'])][:12]
    for c in focus:
        r = pd.to_numeric(mm_c[c], errors='coerce').dropna()
        s = pd.to_numeric(syn_cmp[c], errors='coerce').dropna()
        if r.empty or s.empty:
            continue
        lo = np.nanpercentile(np.concatenate([r.values, s.values]), 1)
        hi = np.nanpercentile(np.concatenate([r.values, s.values]), 99)
        if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
            continue
        bins = np.linspace(lo, hi, 30)
        plt.figure(figsize=(7, 4))
        plt.hist(r.clip(lo, hi), bins=bins, alpha=.55, density=True, label='real')
        plt.hist(s.clip(lo, hi), bins=bins, alpha=.55, density=True, label='synthetic')
        plt.xlabel(c)
        plt.ylabel('Density')
        plt.title(f'Real vs synthetic: {c}')
        plt.legend()
        safe = re.sub(r'[^A-Za-z0-9_-]+', '_', c)[:90]
        savefig(f'real_vs_synthetic_{safe}.png')

    # 9D. Medication prevalence if common med__ indicators exist.
    common_med = [c for c in common if c.lower().startswith('med__')]
    if common_med:
        med_prev_cmp = pd.DataFrame({
            'feature': common_med,
            'real_prevalence': [pd.to_numeric(mm_c[c], errors='coerce').fillna(0).gt(0).mean() for c in common_med],
            'synthetic_prevalence': [pd.to_numeric(syn_cmp[c], errors='coerce').fillna(0).gt(0).mean() for c in common_med],
        })
        med_prev_cmp['abs_diff'] = (med_prev_cmp['real_prevalence'] - med_prev_cmp['synthetic_prevalence']).abs()
        med_prev_cmp.sort_values('abs_diff', ascending=False).to_csv(TAB / 'real_vs_synthetic_medication_prevalence.csv', index=False)

    print('Real-vs-synthetic comparison tables/figures saved.')
else:
    print('Skipped because no synthetic CSV is currently present.')

# ============================================================
# 10. FINAL SUMMARY
# ============================================================
banner('SAVE SUMMARY')
with open(OUT / 'eda_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, default=str)

print(f'All outputs written under: {OUT}')
print('Main folders:')
print('  cleaned/  -> conservative cleaned/flagged CSVs')
print('  tables/   -> quality + EDA reports')
print('  figures/  -> plots')
print('  eda_summary.json -> compact machine-readable summary')



LOAD SOURCE DATA


FileNotFoundError: Missing required file: \mnt\data\labs_deidentified.xlsx